In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [5]:
!pip install emoji==0.6.0 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
# BERTweet trained on 850M tweets, fine-tuned for emotion
# Uses same labels as Cardiff Twitter: 11 emotions
emotion_model = pipeline(
    "text-classification",
    model="finiteautomata/bertweet-base-emotion-analysis",
    top_k=None,
    truncation=True
)

# Quick test
test = emotion_model("I'm so excited for the Super Bowl!")
print(f"Number of labels: {len(test[0])}")
print(f"All labels: {[item['label'] for item in test[0]]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6876.24it/s]


Number of labels: 7
All labels: ['joy', 'others', 'anger', 'surprise', 'sadness', 'fear', 'disgust']


In [7]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:1500] for t in texts]  # BERTweet has 128-token limit, keep truncation aggressive

print(f"Running BERTweet emotion model on {len(texts)} posts...")
results = emotion_model(texts, batch_size=8)
print(f"Got {len(results)} results")

Running BERTweet emotion model on 797 posts...
Got 797 results


In [8]:
# BERTweet output labels
emotion_labels = ["joy", "others", "anger", "surprise", "sadness", "fear", "disgust"]

# Rename columns: locked 6 emotions + "others" (we ignore this later, similar to neutral)
emo_cols = [f"emo_{c}" for c in emotion_labels]

# Drop existing columns if any (safe re-run)
df = df.drop(columns=[c for c in emo_cols + ["dominant_emotion", "dominant_emotion_score"] if c in df.columns])

# Build score dataframe
rows = [{item["label"]: item["score"] for item in res} for res in results]
emo_df = pd.DataFrame(rows)[emotion_labels]
emo_df.columns = emo_cols
emo_df.index = df.index[df["has_text"]]
df = df.join(emo_df)

# Compute dominant emotion (only from our locked 6, ignoring "others")
locked_emo_cols = [f"emo_{e}" for e in ["anger", "disgust", "fear", "joy", "sadness", "surprise"]]

df["dominant_emotion"] = None
df["dominant_emotion_score"] = None
df.loc[df["has_text"], "dominant_emotion"] = (
    df.loc[df["has_text"], locked_emo_cols].idxmax(axis=1).str.replace("emo_", "")
)
df.loc[df["has_text"], "dominant_emotion_score"] = df.loc[df["has_text"], locked_emo_cols].max(axis=1)

print(df[["student_id", "text_source", "dominant_emotion", "dominant_emotion_score"]].head())

  student_id         text_source dominant_emotion dominant_emotion_score
0        1_A  caption+transcript          disgust               0.380506
1        1_A  caption+transcript              joy                0.02292
2        1_A        caption_only              joy               0.015309
3        2_A        caption_only              joy               0.025771
4        2_A        caption_only              joy               0.040377


In [9]:
print("=== Overall dominant emotion counts (from locked 6 only) ===")
print(df["dominant_emotion"].value_counts(dropna=False))
print()
print("=== Mean scores across all posts ===")
print(df[emo_cols].mean().sort_values(ascending=False))
print()
print("=== 'Others' score distribution ===")
print(df["emo_others"].describe())

=== Overall dominant emotion counts (from locked 6 only) ===
dominant_emotion
joy         584
disgust     120
None        108
surprise     34
sadness      25
fear         23
anger        11
Name: count, dtype: int64

=== Mean scores across all posts ===
emo_others      0.693260
emo_joy         0.203568
emo_disgust     0.034584
emo_sadness     0.019852
emo_surprise    0.019747
emo_anger       0.015748
emo_fear        0.013241
dtype: float64

=== 'Others' score distribution ===
count    797.000000
mean       0.693260
std        0.376637
min        0.002827
25%        0.368131
50%        0.931463
75%        0.970538
max        0.980730
Name: emo_others, dtype: float64


In [10]:
# Load LLM scores for comparison
llm_emo_obj = pd.read_csv("../Outputs/llm_emotion_objective_905.csv")

# Load master to get Cardiff dominant
master = pd.read_csv("../Outputs/routed_master_905.csv")

LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Get dominant emotion per source, for yellow-bucket posts only (where LLM has data)
def dominant(row, prefix, emotions=LOCKED_EMOTIONS):
    scores = {e: row[f"{prefix}_{e}"] for e in emotions if f"{prefix}_{e}" in row}
    if not scores:
        return None
    return max(scores, key=scores.get)

# Only look at yellow posts (where we have LLM data)
yellow_mask = master["emotion_bucket"] == "yellow"
yellow_master = master[yellow_mask].copy()
yellow_bertweet = df.loc[yellow_master.index]

# Cardiff dominant (from master)
yellow_master["cardiff_dom"] = yellow_master.apply(lambda r: dominant(r, "cardiff"), axis=1)

# BERTweet dominant (from bertweet df)
yellow_master["bertweet_dom"] = yellow_bertweet["dominant_emotion"].values

# LLM dominant (need to merge)
llm_renamed = llm_emo_obj.rename(columns={e: f"llm_{e}" for e in LOCKED_EMOTIONS})
merged = yellow_master.merge(
    llm_renamed[[f"llm_{e}" for e in LOCKED_EMOTIONS] + ["_post_index"]],
    left_index=True,
    right_on="_post_index",
    how="inner"
)
merged["llm_dom"] = merged.apply(lambda r: dominant(r, "llm"), axis=1)

print(f"Yellow posts with all models: {len(merged)}\n")

# Pairwise agreement
print("=== Pairwise dominant emotion agreement (yellow posts only) ===")
pairs = [
    ("Cardiff", "BERTweet", "cardiff_dom", "bertweet_dom"),
    ("Cardiff", "LLM", "cardiff_dom", "llm_dom"),
    ("BERTweet", "LLM", "bertweet_dom", "llm_dom"),
]
for name1, name2, col1, col2 in pairs:
    agree = (merged[col1] == merged[col2]).sum()
    pct = agree / len(merged) * 100
    print(f"  {name1:10s} vs {name2:10s}: {agree:4d}/{len(merged)} ({pct:.1f}%)")

# Also: on yellow posts, does BERTweet resolve the transformer-transformer disagreement?
# If yes, fewer posts would be yellow
print("\n=== BERTweet's agreement with each existing transformer on yellow posts ===")
existing = master.loc[yellow_master.index]

for model in ["cardiff", "distil", "goemo"]:
    existing_dom = existing.apply(lambda r: dominant(r, model), axis=1)
    bertweet_dom = yellow_bertweet["dominant_emotion"].values
    agree = (existing_dom.values == bertweet_dom).sum()
    pct = agree / len(existing) * 100
    print(f"  BERTweet vs {model:10s}: {agree}/{len(existing)} ({pct:.1f}%)")

Yellow posts with all models: 696

=== Pairwise dominant emotion agreement (yellow posts only) ===
  Cardiff    vs BERTweet  :  476/696 (68.4%)
  Cardiff    vs LLM       :  474/696 (68.1%)
  BERTweet   vs LLM       :  457/696 (65.7%)

=== BERTweet's agreement with each existing transformer on yellow posts ===
  BERTweet vs cardiff   : 476/696 (68.4%)
  BERTweet vs distil    : 278/696 (39.9%)
  BERTweet vs goemo     : 251/696 (36.1%)


In [11]:
output_path = "../Outputs/emotion_bertweet_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/emotion_bertweet_905.csv
